In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw12.ipynb")

# HW 12: Modeling Cook County Housing Prices

Apply the modeling concepts from **Lectures 22–23** to the **Cook County housing dataset**.

## Overview

Complete **12 questions**, including **two visualization tasks**, covering modeling, loss functions, and model evaluation. No custom functions or written responses are required.

## Learning Objectives

- Choose a model, define a loss function, and fit parameters analytically.
- Compare constant and linear models.
- Evaluate predictions and inspect residuals.
- Examine the effects of outliers.
- Fit and evaluate a log-response model.

## Instructions

- Complete the questions in order.
- Keep full precision throughout your calculations.
- **Interpret metrics as descriptions of the observed sample.** They do not measure performance on unseen data.

---
<a id="question-outline"></a>
## Question Outline

- [Question 1: Modeling Arrays](#q1) — 6 points
- [Question 2: Mean and Median Constants](#q2) — 8 points
- [Question 3: Candidate Linear Model](#q3) — 8 points
- [Question 4: Analytical Linear Regression](#q4) — 10 points
- [Question 5: MSE, MAE, and RMSE](#q5) — 8 points
- [Question 6: Estimating Equations](#q6) — 8 points
- [Question 7: Prediction Plots](#q7) — 10 points (manual visualization)
- [Question 8: Model Comparison](#q8) — 8 points
- [Question 9: Outlier Sensitivity](#q9) — 8 points
- [Question 10: Log-Response Model](#q10) — 8 points
- [Question 11: Back-Transformed Predictions](#q11) — 8 points
- [Question 12: Residual Patterns](#q12) — 10 points (manual visualization)

## Before You Start

- **Enter your answers in the designated answer cell(s)** directly below each question.
- **Do not add or delete any cells in this notebook.** Changing the cell structure can cause formatting errors on Gradescope and prevent successful autograding.
- Complete all work in the provided answer cell(s), keeping every cell in its original order.
- **Your solutions will be evaluated using both public and hidden tests.** Public tests are available in the notebook to help you check your work. Hidden tests run after you submit your notebook to Gradescope and may check additional cases and requirements. **Passing all public tests does not guarantee full credit.**. Check your results carefully and follow every requirement in each question.

### Debugging Guide
If you run into any technical issues, we highly recommend checking out the look at the [Debugging Guide](https://mtu.instructure.com/courses/1619791/pages/debugging-guide). In this guide, you can find general questions about Jupyter notebooks / Datahub, Gradescope, common `pandas` errors, and more

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

import zipfile
import os

# Plot settings
plt.rcParams['figure.figsize'] = (12, 9)
plt.rcParams['font.size'] = 12

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

# Dataset Overview: Cook County Housing Sales

This assignment uses housing data from **Cook County, Illinois**, which includes Chicago. The source dataset contains more than **500,000 records**, with **61 predictor variables** and the response variable, **`Sale Price`**.

Variable descriptions are provided in **`data/codebook.txt`**. To view the codebook, open the `data` folder in the notebook’s file browser and select `codebook.txt`. Selected columns have been excluded to keep data cleaning and preparation within the scope of this assignment.

The provided data are divided into:

- **Training set:** 204,792 observations
- **Test set:** 68,264 observations

**This homework uses only the training set.** All model fitting and evaluation will therefore describe the observed training sample rather than performance on unseen data.

## Load the Data

The CSV files are stored in **`cook_county_data.zip`** to reduce file size. **Run the following cells in order** to extract and load the data.

In [ ]:
if not (os.path.exists('cook_county_train.csv')):
    with zipfile.ZipFile('data/cook_county_data.zip') as item:
        item.extractall()

Let's load the initial data.

In [ ]:
initial_data = pd.read_csv("cook_county_train.csv", index_col='Unnamed: 0')

As a good sanity check, we should at least verify that the data shape matches the description.

In [ ]:
# 204,792 observations and 62 features in training data
assert initial_data.shape == (204792, 62)
# Sale Price is provided in the training data
assert 'Sale Price' in initial_data.columns.values

The next order of business is getting a feel for the variables in our data.  A more detailed description of each variable is included in `codebook.txt` (in the same directory as this notebook).  **You should take some time to familiarize yourself with the codebook before moving forward.**

Let's take a quick look at all the current columns in our initial data.

In [ ]:
initial_data.columns.values

In [ ]:
initial_data.head()

In [ ]:
initial_data['Description'][0]

---
<a id="q1"></a>
## Question 1: Prepare the Modeling Arrays (6 points)

Select `Building Square Feet` and `Sale Price` into `housing`, keeping records with positive building area and sale price at least 1000 dollars. Remove rows missing either value and retain original row order.

Create NumPy arrays `x` = area divided by 1000 and `y` = price divided by 1000. Thus, **x is in thousands of square feet and y is in thousands of dollars**. Use these arrays throughout the homework; leave `initial_data` unchanged.

In [ ]:
keep = ...
housing = ...
x = ...
y = ...

In [ ]:
grader.check("q1")

[↑ Return to question outline](#question-outline)

---
<a id="q2"></a>
## Question 2: Compare Mean and Median Constants (8 points)

A constant model predicts the same sale price for every property. Compare two constant models using the response array `y`.

1. Set `mean_constant` to the mean of `y` and `median_constant` to the median of `y`.
2. Create a DataFrame named `constant_comparison` with:
   - **Rows:** `Mean`, `Median`, in that order.
   - **Columns:** `MSE`, `MAE`, in that order.
3. For each constant, calculate the mean squared error (MSE) and mean absolute error (MAE) using **all values in `y`**.

**Recall:** The mean minimizes MSE among constant models, while the median minimizes MAE. Keep full precision in your calculations.

In [ ]:
mean_constant = ...
median_constant = ...
constant_comparison = pd.DataFrame({
    'MSE': [..., ...],
    'MAE': [..., ...]
}, index=['Mean', 'Median'])

In [ ]:
grader.check("q2")

[↑ Return to question outline](#question-outline)

---
<a id="q3"></a>
## Question 3: Evaluate a Candidate Linear Model (8 points)

Evaluate the candidate model:

$$
\hat{y} = 50 + 100x
$$

Here, `x` is building area in thousands of square feet, and predicted prices are in thousands of dollars.

1. Create `candidate_predictions` using the candidate model.
2. Calculate `candidate_residuals` as **observed minus predicted**: `y - candidate_predictions`.
3. Create a pandas Series named `candidate_metrics` with the following entries, in order:
   - `signed_error`: mean residual.
   - `MSE`: mean squared residual.
   - `MAE`: mean absolute residual.

**Note:** Positive and negative residuals can cancel, so a small mean signed error does not necessarily indicate accurate predictions.

In [ ]:
candidate_predictions = ...
candidate_residuals = ...
candidate_metrics = pd.Series(
    [..., ..., ...],
    index=['signed_error', 'MSE', 'MAE']
)

In [ ]:
grader.check("q3")

[↑ Return to question outline](#question-outline)

---
<a id="q4"></a>
## Question 4: Fit Simple Linear Regression Analytically (10 points)

Fit a least-squares regression line to predict sale price from building area using the analytical formulas from **Lecture 22**.

1. Calculate the Pearson correlation between `x` and `y`, and store it in `r`.
2. Calculate the slope, `theta1_hat`, and intercept, `theta0_hat`:

   $$
   \hat{\theta}_1 = r\frac{\operatorname{std}(y)}{\operatorname{std}(x)},
   \qquad
   \hat{\theta}_0 = \operatorname{mean}(y) - \hat{\theta}_1\operatorname{mean}(x)
   $$

3. Create `linear_predictions` for all observations:

   $$
   \hat{y} = \hat{\theta}_0 + \hat{\theta}_1x
   $$

**Use population standard deviations (`ddof=0`) and retain full precision. Do not use a regression fitting library or numerical optimizer.**

In [ ]:
r = ...
theta1_hat = ...
theta0_hat = ...
linear_predictions = ...

In [ ]:
grader.check("q4")

[↑ Return to question outline](#question-outline)

---
<a id="q5"></a>
## Question 5: Calculate MSE, MAE, and RMSE (8 points)

Evaluate the fitted linear model using `linear_predictions` from Question 4.

1. Create `fitted_residuals` as **observed minus predicted**: `y - linear_predictions`.
2. Create a pandas Series named `linear_metrics` containing the following metrics, in order:
   - `MSE`: mean squared residual.
   - `MAE`: mean absolute residual.
   - `RMSE`: square root of the mean squared residual.

**Units:** MAE and RMSE are measured in **thousands of dollars**. MSE is measured in **(thousands of dollars)²**. Retain full precision.

In [ ]:
fitted_residuals = ...
linear_metrics = pd.Series({
    'MSE': ...,
    'MAE': ...,
    'RMSE': ...
})

In [ ]:
grader.check("q5")

[↑ Return to question outline](#question-outline)

---
<a id="q6"></a>
## Question 6: Check the Estimating Equations (8 points)

For a least-squares regression line with an intercept, the residuals satisfy two estimating equations: the mean residual and the mean of the predictor–residual products are both zero, up to numerical precision.

Using `x` and `fitted_residuals`:

1. Calculate `mean_residual`, the mean of the residuals.
2. Calculate `mean_x_residual`, the mean of the elementwise products `x * fitted_residuals`.
3. Set `normal_equations_hold` to a Boolean indicating whether **both values are close to zero**, using an absolute tolerance of **`1e-8`**.

**Note:** A mean residual near zero does not imply that individual prediction errors are small.

In [ ]:
mean_residual = ...
mean_x_residual = ...
normal_equations_hold = ...

In [ ]:
grader.check("q6")

[↑ Return to question outline](#question-outline)

<!-- BEGIN QUESTION -->

---
<a id="q7"></a>
## Question 7: Visualize Constant and Linear Predictions (10 points)

Create a single plot comparing the observed sale prices with the constant and fitted linear models.

1. Create a figure named `model_figure`.
2. Plot every 50th observation using `x[::50]` and `y[::50]` to reduce clutter.
3. Overlay both model predictions:
   - The fitted regression line across the full observed range of `x`, using `theta0_hat` and `theta1_hat`.
   - A horizontal line at `mean_constant`.
4. Label both axes with their units, add a descriptive title and legend, and display the plot.

**Use the model parameters calculated from all observations. Subsample only the scatterplot points.**

**Manual grading rubric (10 points):**

- Correct scatterplot and axis units: **3 points**
- Correct fitted regression line and mean constant line: **4 points**
- Descriptive title, legend, and readable display: **3 points**

In [ ]:
model_figure, ax = plt.subplots(figsize=(7,4))
# Scatterplot, fitted line, and constant prediction.
...
# Labels, title, and legend.
...
plt.show()

<!-- END QUESTION -->

[↑ Return to question outline](#question-outline)

---
<a id="q8"></a>
## Question 8: Compare Models on the Same Scale (8 points)

Compare the mean constant model and the fitted linear model using **all observations** and the original response array `y`.

1. Create a DataFrame named `model_comparison` with:
   - **Rows:** `Constant`, `Linear`, in that order.
   - **Columns:** `MSE`, `RMSE`, in that order.
2. Calculate `rmse_reduction` as the constant model’s RMSE minus the linear model’s RMSE.
3. Set `best_model` to the row label with the lowest MSE. If the values are tied, select the first row.

**Use predictions in thousands of dollars.** A positive `rmse_reduction` indicates that the linear model has a lower RMSE on the observed sample.

In [ ]:
model_comparison = pd.DataFrame({
    'MSE': [..., ...]
}, index=['Constant', 'Linear'])
model_comparison['RMSE'] = ...
rmse_reduction = ...
best_model = ...

In [ ]:
grader.check("q8")

[↑ Return to question outline](#question-outline)

---
<a id="q9"></a>
## Question 9: Measure Outlier Sensitivity (8 points)

Examine how adding one extreme sale price affects the mean and median constant models.

1. Create `y_with_outlier` by appending **10 times the maximum value of `y`** to a new array. Leave `y` unchanged.
2. Create a pandas Series named `constant_shifts`, indexed `Mean` and `Median`, in that order. Calculate each shift as **the new constant minus its original value**:
   - `Mean`: mean of `y_with_outlier` minus `mean_constant`.
   - `Median`: median of `y_with_outlier` minus `median_constant`.
3. Set `more_sensitive` to the label with the **larger absolute shift**. If the shifts are tied, select the first label.

This comparison examines the outlier sensitivity of the constants that minimize MSE and MAE.

In [ ]:
y_with_outlier = ...
constant_shifts = pd.Series(
    [..., ...],
    index=['Mean', 'Median']
)
more_sensitive = ...

In [ ]:
grader.check("q9")

[↑ Return to question outline](#question-outline)

---
<a id="q10"></a>
## Question 10: Fit a Log-Response Model (8 points)

Following **Lecture 23**, investigate an alternative model by transforming the response variable while keeping the predictor `x` unchanged.

1. Store the natural logarithm of `y` in `log_y`.
2. Fit a least-squares line predicting `log_y` from `x`. Calculate the coefficients analytically and store them in `log_slope` and `log_intercept`.
3. Store the fitted values for all observations in `log_predictions`.

**Requirements:**

- Use population standard deviations (`ddof=0`).
- Retain full precision.
- Do not use a regression fitting library or numerical optimizer.
- Keep predictions on the log-response scale; do not transform them back in this question.

In [ ]:
log_y = ...
log_slope = ...
log_intercept = ...
log_predictions = ...

In [ ]:
grader.check("q10")

[↑ Return to question outline](#question-outline)

---
<a id="q11"></a>
## Question 11: Back-Transform and Compare Predictions (8 points)

Evaluate the log-response model on the same response scale as the original linear model.

1. Apply the inverse of the natural logarithm to `log_predictions` and store the results in `back_predictions`.
2. Evaluate these predictions against `y`. Store the RMSE in `back_rmse` and the MAE in `back_mae`.
3. Compare the two models’ RMSE values. Set `lower_rmse_model` to `'Linear'` or `'Log-response'`, selecting `'Linear'` if they are tied.

**Compare both models in thousands of dollars.** Retain full precision and use all observations.

**Note:** This simple back-transformation does not generally provide an unbiased estimate of the conditional mean sale price.

In [ ]:
back_predictions = ...
back_rmse = ...
back_mae = ...
lower_rmse_model = ...

In [ ]:
grader.check("q11")

[↑ Return to question outline](#question-outline)

<!-- BEGIN QUESTION -->

<a id="q12"></a>
## Question 12: Inspect Residual Patterns (10 points)

Create `residual_figure` with two side-by-side scatterplots, using every 50th observation. On both x-axes use `x`. On the left show `y - linear_predictions`; on the right show `log_y - log_predictions`. Add a horizontal zero line, title, and descriptive axis labels to each panel. Use the correct response scale in each residual label. Do not compare residual magnitudes directly across these different scales.

**Manual rubric (10 points):** correct two residual arrays and predictor (4); zero lines (2); scale-specific labels and titles (2); readable layout and display (2).

---
<a id="q12"></a>
## Question 12: Inspect Residual Patterns (10 points)

Create a figure named `residual_figure` containing two side-by-side scatterplots to examine residual patterns for the two fitted models.

- **Left panel:** Linear model residuals against building area.
- **Right panel:** Log-response model residuals against building area.

Use every **50th observation** in both panels. Calculate residuals as **observed minus predicted**, using each model’s response scale.

For each panel:

- Add a horizontal reference line at zero.
- Include a descriptive title.
- Label the horizontal axis with building area and its units.
- Label the vertical axis with the appropriate residual scale.

Display both panels in a readable layout.

**Note:** The residuals use different response scales. Do not compare their magnitudes directly.

**Manual grading rubric (10 points):**

- Correct residuals, predictor, and subsampling in both panels: **4 points**
- Horizontal zero lines in both panels: **2 points**
- Accurate axis labels, response scales, and titles: **2 points**
- Readable layout and displayed figure: **2 points**

In [ ]:
residual_figure, axes = plt.subplots(1,2,figsize=(11,4))
...

<!-- END QUESTION -->

[↑ Return to question outline](#question-outline)

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Congratulations on finishing HW 12!

What's next? In Project 3, you will focus on building a linear regression model to predict house prices. You will be well-prepared to build such a model: you have considered what is in this dataset, what it can be used for, and engineered some features that should be useful for prediction. As we look into creating a house-pricing model for Cook County, we'll also consider some challenging social implications that this model may carry. 


### Instructions


Run all cells, review the test results, and display the two required figures (Q7 and Q12). 
Below, you will see a cell. Save before running the generated export cell. Running this cell will automatically generate a zip file with your autograded answers. 

The instructor source includes solutions and hidden tests. Distribute the student version generated by Otter Assign.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)